# ZEV mandate tracker

How each manufacturer is doing against this year's ZEV mandate target, from the latest VES registrations.

- **Data:** `zevm.data` pulls from BigQuery and caches for 7 days. Use `refresh=True` in `prepare` to force a new pull (about $0.14 for cars).
- **Recent months** are scaled up for records VES is still adding (late-data uplift). **This year** is estimated to full year from seasonality. The fitted figures are shown under `notes`.
- **Rules** come from `params/vets.yaml` (the law as amended). How the accounting works is documented at the top of `zevm/compliance.py`.

In [ ]:
import pandas as pd
from IPython.display import display

from zevm import compliance, plots, rules, scenarios
from zevm.scenarios import Scenario, Strategy

plots.use_style()
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
VEHICLE = "car"          # "car" or "van"
PARAMS = rules.load("vets")

h = scenarios.prepare(VEHICLE, params=PARAMS)
YEAR = int(h.yearly["year"].max())
h.notes

## This scheme year (full-year estimate)

Runs 2024 to this year with the real banking and borrowing history, then shows this year. With `Strategy()`, all flexibilities and trading are on.

In [ ]:
tracker = scenarios.run(h, Scenario("Tracker", params=PARAMS, end_year=YEAR))
summary = scenarios.market_summary(tracker)
summary[["year", "basis", "zev_share", "zev_target", "required_after_co2", "co2_credits", "traded", "borrowed",
         "bank_end", "debt_end", "shortfall", "payment_gbp_m", "paying"]]

In [ ]:
plots.zev_share_monthly(h.monthly, PARAMS, VEHICLE);

In [ ]:
plots.position_bars(tracker, YEAR, top_n=20);

In [ ]:
plots.position_bars(tracker, YEAR, top_n=20, per_vehicle=True,
                    title=f"Surplus / shortfall as a share of registrations, {YEAR} (estimate)");

In [ ]:
cols = ["group", "total_registrations", "zev_share", "required_after_co2", "co2_baseline", "co2_credits", "co2_credit_cap",
        "banked_used", "bought", "sold", "borrowed", "repaid", "shortfall", "bank_end", "debt_end", "payment_gbp"]
this_year = tracker[tracker["year"] == YEAR].assign(
    required_after_co2=lambda d: d["required_zev_share_after_co2"]).sort_values("total_registrations", ascending=False)
this_year[cols].head(30)

## Rolling 12 months

The last 12 months for each participant, measured against this year's target on their own: no bank, borrowing or trading. This is the view the old tracker gave.

In [ ]:
trail = scenarios.trailing_inputs(h)
rolling = compliance.run(trail.drop(columns="basis"), PARAMS, VEHICLE, Strategy(trading="none", banking=False, borrowing=False))
rolling["basis"] = trail["basis"].iloc[0]
plots.position_bars(rolling, YEAR, top_n=20, title=f"Last 12 months vs {YEAR} target, own allowances and CO2 credits only");

## Export

In [ ]:
from pathlib import Path
out = Path("../outputs"); out.mkdir(exist_ok=True)
this_year[cols].to_csv(out / f"tracker_{VEHICLE}_{h.as_of:%Y-%m}.csv", index=False)
summary.to_csv(out / f"tracker_{VEHICLE}_market_{h.as_of:%Y-%m}.csv", index=False)